In [45]:
import json
import glob

t = glob.glob("raw_data/*.json")
t[:5], len(t)

(['raw_data\\brave-search.json',
  'raw_data\\fetch.json',
  'raw_data\\filesystem.json',
  'raw_data\\github.json',
  'raw_data\\google-maps.json'],
 9)

In [46]:
with open(t[0], "r") as f:
    data = json.load(f)
data.keys()

dict_keys(['tools', 'resources'])

In [47]:
filter_prompt = """
You are an expert with a deep understanding of user asks, You have the ability to determine what user asks can be completed provided a set of tools.
You have been provided a list of tools and their descriptions. Your task is to determine if any user ask can be created using the tools provided and provide an example of such a user ask if yes.

The tools provided are:
{tool_list}
Provide your answer in the this format:
<explanation>[Explaining Why the answer was chosen]</explanation>
<answer>[yes/no]</answer>
<example_user_ask>[An example of a user ask that can be completed with the said tools (if yes)]</example_user_ask>
"""

In [48]:
import anthropic
import os
from dotenv import main

main.load_dotenv()

client = anthropic.Anthropic(
    api_key=os.environ.get("ANTHROPIC_KEY"),
)

def return_response(prompt):
    message = client.messages.create(
        model="claude-3-7-sonnet-20250219",
        max_tokens=8192,
        messages=[
            {"role": "user", "content": prompt},
        ]
    )

    return message.content[0].text.strip()

In [49]:
all_tool_data = {}
for f in t:
    with open(f, "r") as file:
        data = json.load(file)
        all_tool_data[f.split(".json")[0].split("\\")[1]] = data

In [30]:
import random

def parse_response(response):
    """
    Parse the response from the model and return the task and answer.
    """
    try:
        task = response.split("<example_user_ask>")[1].split("</example_user_ask>")[0].strip()
        answer = response.split("<answer>")[1].split("</answer>")[0].strip()
        explanation = response.split("<explanation>")[1].split("</explanation>")[0].strip()
        return task, answer, explanation
    except Exception as e:
        print(f"Error parsing response: {e}")
        return None, None, None

def sample_individual_server(id):
    server_data = all_tool_data[id]
    ## sample 1 to 5 tools from the server.
    num_tools = random.randint(1, 5)
    num_tools = min(num_tools, len(server_data["tools"]))
    sampled_tools = random.sample(server_data["tools"], num_tools)
    ## use filter_prompt to get a user ask.
    filtered_prompt = filter_prompt.format(tool_list=str(sampled_tools))
    response = return_response(filtered_prompt)
    ## parse the response and return the task and answer.
    task, answer, explanation = parse_response(response)
    ## return the task, answer, explanation and the sampled tool names.
    sampled_tool_names = [tool["name"] for tool in sampled_tools]
    ## make sampled_tool_names a string.
    sampled_tool_names = ", ".join(sampled_tool_names)
    return task, answer, explanation, sampled_tool_names

In [31]:
sample_individual_server("brave-search")

('Find me the top 5 Italian restaurants near me',
 'yes',
 'The provided tool is "brave_local_search" which can search for local businesses and services. If no local results are found, it automatically falls back to web search. The tool requires a query string and optionally takes a count parameter to specify the number of results (maximum 20).\n\nThis tool can handle user requests related to:\n1. Finding local businesses (restaurants, shops, services, etc.)\n2. Getting information about nearby places\n3. Searching for local services\n4. General web searches (as it falls back to web search if no local results are found)\n\nGiven these capabilities, a user could ask for various local search-related information that this tool could fulfill.',
 'brave_local_search')

In [41]:
import pandas as pd

def process_server(id):
    ## get number of tools in the server.
    df_dict_arr = []
    server_data = all_tool_data[id]
    num_tools = len(server_data["tools"])
    ## get ceil(len(server_data["tools"]) / 5) * 2 number of tasks.
    num_tasks = (num_tools // 5) * 2 + 2
    num_tasks = min(num_tasks, 10)
    ## sample num_tasks number of tasks from the server.
    for i in range(num_tasks):
        task, answer, explanation, sampled_tool_names = sample_individual_server(id)
        ## save the task, answer, explanation and the sampled tool names to a file.
        df_dict_arr.append({
            "task": task,
            "answer": answer,
            "explanation": explanation,
            "sampled_tool_names": sampled_tool_names,
            "server_id": id
        })
        print(f"Task: {task} for server {id} with tools {sampled_tool_names}")
     
    # print(f"Processed {id} with {len(df_dict_arr)} tasks.")
    ## create a dataframe from the list of dictionaries.
    df = pd.DataFrame(df_dict_arr)
    ## save the dataframe to a tsv file.
    df.to_csv(f"raw_data/single-server-qs/{id}.tsv", sep="\t", index=False)

In [52]:
ids = list(all_tool_data.keys())
ids[:5], len(ids)

(['brave-search', 'fetch', 'filesystem', 'github', 'google-maps'], 9)

In [53]:
for id in ids:
    process_server(id)

Task: Find pizza restaurants near me and show me the top 5 results. for server brave-search with tools brave_local_search, brave_web_search
Task: Find Italian restaurants near me, and if there aren't any, search for Italian food recipes I can make at home. for server brave-search with tools brave_local_search, brave_web_search
Task: Can you retrieve and show me the content of the Wikipedia page about artificial intelligence? Please format it nicely so it's easy to read. for server fetch with tools fetch
Task: Can you fetch the main content from the Wikipedia page about artificial intelligence and summarize the key concepts for me? for server fetch with tools fetch
Task: Could you create a new text file called "notes.txt" with my meeting agenda in it, then check which directories are available, and finally rename the file to "meeting_agenda.txt"? for server filesystem with tools write_file, list_allowed_directories, move_file
Task: Can you read the content of my "notes.txt" and "todo.tx

In [50]:
all_tool_data.keys(), len(all_tool_data.keys())

(dict_keys(['brave-search', 'fetch', 'filesystem', 'github', 'google-maps', 'postgres', 'puppeteer', 'sentry', 'slack']),
 9)

In [51]:
all_tool_data["filesystem"].keys()

dict_keys(['resources', 'tools'])

In [65]:
all_tool_flat_list = [all_tool_data[key] for key in all_tool_data.keys()]
all_tool_flat_list = [item for sublist in all_tool_flat_list for item in sublist["tools"]]

def sample_multiple_servers():
    ## sample 1 to 5 servers from all_tool_data.
    num_servers = random.randint(1, 5)
    num_tools = random.randint(num_servers, 5)
    ## distribute the number of tools to the servers.
    server_ids = random.sample(list(all_tool_data.keys()), num_servers)
    server_tool_counts = [random.randint(1, num_tools // num_servers) for _ in range(num_servers)]
    ## make sure the sum of server_tool_counts is equal to num_tools.
    while sum(server_tool_counts) < num_tools:
        server_tool_counts[random.randint(0, num_servers - 1)] += 1
    sampled_tools = []
    for i in range(num_servers):
        server_id = server_ids[i]
        server_data = all_tool_data[server_id]
        ## sample the number of tools from the server.
        num_tools = min(server_tool_counts[i], len(server_data["tools"]))
        sampled_tools += random.sample(server_data["tools"], num_tools)
    ## use filter_prompt to get a user ask.
    filtered_prompt = filter_prompt.format(tool_list=str(sampled_tools))
    response = return_response(filtered_prompt)
    ## parse the response and return the task and answer.
    task, answer, explanation = parse_response(response)
    ## return the task, answer, explanation and the sampled tool names.
    sampled_tool_names = [tool["name"] for tool in sampled_tools]
    ## make sampled_tool_names a string.
    sampled_tool_names = ", ".join(sampled_tool_names)
    return task, answer, explanation, sampled_tool_names

In [67]:
import tenacity

@tenacity.retry(
    wait=tenacity.wait_random(min=180, max=240),  # wait between 3 to 4 minutes
    stop=tenacity.stop_after_attempt(5)           # stop after 5 attempts
)
def sample_multiple_servers_with_retry():
    return sample_multiple_servers()

In [68]:
def process_multiple_servers():
    num_tasks = len(all_tool_flat_list) // 5 * 2 + 2
    ## sample num_tasks number of tasks from the server.
    df_dict_arr = []
    for i in range(num_tasks):
        task, answer, explanation, sampled_tool_names = sample_multiple_servers_with_retry()
        ## save the task, answer, explanation and the sampled tool names to a file.
        df_dict_arr.append({
            "task": task,
            "answer": answer,
            "explanation": explanation,
            "sampled_tool_names": sampled_tool_names,
            "server_id": "multiple_servers"
        })
        print(f"Task: {task} for server multiple_servers with tools {sampled_tool_names}")
    ## create a dataframe from the list of dictionaries.
    df = pd.DataFrame(df_dict_arr)
    ## save the dataframe to a tsv file.
    df.to_csv(f"raw_data/multiple-server-qs/collated.tsv", sep="\t", index=False)

In [69]:
process_multiple_servers()

Task: Could you take a screenshot of the homepage of wikipedia.org, fetch the content from the site, and also tell me the file size of the screenshot once it's saved? for server multiple_servers with tools get_file_info, puppeteer_screenshot, fetch
Task: Can you pull a list of all customers from our database who made purchases in the last month, then check if any of them are also users in our Slack workspace? After that, navigate to our company dashboard to verify the total sales numbers. for server multiple_servers with tools query, slack_get_users, puppeteer_navigate
Task: Can you query our database to find the top 5 customers by total purchase amount, and then add a thumbs-up reaction to the Slack message where our team was discussing the quarterly sales report? The message is in the #sales channel with timestamp 1652784562.369. for server multiple_servers with tools query, slack_get_thread_replies, slack_get_user_profile, slack_add_reaction
Task: Could you please fetch the content 

In [57]:
len(all_tool_flat_list), len(all_tool_flat_list) // 5 * 2 + 2, len(all_tool_flat_list) // 5 * 2 + 2 > 10

(63, 26, True)